# FloodRisk is a classification problem.
# This notebook demonstrates Logistic Regression
# for predicting flood risk categories.

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
#Load Dataset

df = pd.read_csv('../data/raw/flood_data.csv')

In [ ]:
# Select features and target

X = df[['rainfall', 'humidity', 'temperature']]
y = df['flood_Risk']

In [ ]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Majority Class Baseline

baseline = DummyClassifier(strategy='most_frequent')

baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)
baseline_probs = baseline.predict_proba(X_test)[:, 1]

In [ ]:
# Train Logistic Regression Model

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

In [ ]:
# Make Predictions

predictions = pipeline.predict(X_test)

probabilities = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate Model

accuracy = accuracy_score(y_test, predictions)
auc = roc_auc_score(y_test, probabilities)

print("Accuracy:", accuracy)
print("ROC-AUC:", auc)

print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

In [ ]:
# Baseline Comparison

print("Baseline Accuracy:",
      accuracy_score(y_test, baseline_preds))

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, predictions))

In [ ]:
# Cross-Validation

cv_auc = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

print("CV ROC-AUC Scores:", cv_auc)
print("Mean ROC-AUC:", cv_auc.mean())

In [ ]:
# Coefficients Interpretation

model = pipeline.named_steps['model']

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Odds Ratio': np.exp(model.coef_[0])
})

print(coefficients)

In [ ]:
# Confusion Matrix

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions
)